# opslab, in your browser

Four things a life-and-pensions back office needs from its data, and that
general-purpose libraries answer badly. Nothing here needs installing beyond
the package itself: no numpy, scipy, pandas or lifelines.

Run the cells in order. The whole notebook takes about a minute.

## 1. Install

The sample event log, case table and Power BI model travel inside the package,
so there is no data to download.

In [ ]:
%pip install -q git+https://github.com/pr1317/opslab2


## 2. The whole thing, in one command

`opslab try` runs all four modules against the bundled sample and writes a
single self-contained HTML report. The cell below renders it inline.

In [ ]:
!opslab try --out try_out


In [ ]:
from IPython.display import HTML

HTML(open('try_out/report.html').read())


## 3. The process, as the log actually ran it

A directly-follows graph discovered from timestamps. Green marks where cases
start, red where they finish, and the dashed red edges point backwards - which
in a case-management process is exactly the rework.

In [ ]:
from opslab import data
from opslab.calendar import BusinessCalendar
from opslab.eventlog import EventLog
from opslab.processmining import discover_dfg, to_svg

log = EventLog.from_csv(data.events_path())
graph = discover_dfg(log, BusinessCalendar())

HTML(to_svg(graph, min_edge_frequency=5))


### Working hours, not wall-clock

Every duration above is measured on a business calendar. A case that arrives
at 16:30 on Friday and is picked up at 09:30 on Monday waited one hour, not
sixty-five - and an SLA clock that says sixty-five will have you staffing
against a queue that does not exist.

In [ ]:
from datetime import datetime

calendar = BusinessCalendar()
friday_evening = datetime(2026, 3, 6, 16, 30)
monday_morning = datetime(2026, 3, 9, 9, 30)

print('wall clock    : %.1f hours' % ((monday_morning - friday_evening).total_seconds() / 3600))
print('working hours : %.1f hours' % calendar.working_hours_between(friday_evening, monday_morning))


## 4. The open cases are the slow ones

About a tenth of the sample had not finished by the extract date. Dropping
them and taking a median of what is left conditions on the case having already
finished, which systematically excludes the slow ones. Kaplan-Meier keeps them
as evidence up to the point they were last seen.

In [ ]:
from opslab.eventlog import CaseTable
from opslab.sla import kaplan_meier

cases = CaseTable.from_csv(data.cases_path())
durations = [c.duration_hours for c in cases]
events = [1 if c.resolved else 0 for c in cases]

closed = sorted(c.duration_hours for c in cases if c.resolved)
curve = kaplan_meier(durations, events)

print('%d of %d cases were still open at the extract' % (events.count(0), len(cases)))
print('median, closed cases only : %.1f working hours' % closed[len(closed) // 2])
print('median, Kaplan-Meier      : %.1f working hours' % curve.median)


## 5. Checking the estimator, not admiring it

The sample is synthetic for one reason: the coefficients that generated it are
published, so a fitted Cox model can be checked against the truth. On a real
extract you can only report a number and hope.

In [ ]:
from opslab.sla import design_matrix, fit_cox
from opslab.simulate.generator import GROUND_TRUTH_BETA

matrix = design_matrix(
    cases,
    numeric=['complexity', 'backlog_index', 'awaiting_third_party'],
    binary={'priority': 'Urgent', 'channel': 'Post'},
)
model = fit_cox(durations, events, matrix.rows, names=matrix.names)

print('%-24s %9s %9s %s' % ('covariate', 'fitted', 'true', 'inside 95% CI'))
for name, fitted, (low, high) in zip(
    model.result.names, model.result.coefficients, model.result.confidence_intervals()
):
    truth = GROUND_TRUTH_BETA[name]
    print('%-24s %9.4f %9.4f %s' % (name, fitted, truth, 'yes' if low <= truth <= high else 'no'))


## 6. Is this week's number a change, or noise?

A backlog surge was injected into the sample part way through. A threshold
dashboard would have flagged the first bad week and then cried wolf every
week after. The Nelson rules distinguish a shift from ordinary variation.

In [ ]:
!opslab spc --cases try_out/sample_cases.csv


## 7. The model that reports all of this

The bundled Power BI model is written to fail every rule at least once, so
the output is worth reading. Point it at your own `.bim`, `.tmdl` or PBIP
folder to lint that instead.

In [ ]:
from opslab.daxlint import lint, load_model

model = load_model(data.model_path())
findings = lint(model)
print('%s: %d findings across %d measures\n' % (model.name, len(findings), len(model.measures())))
for finding in findings[:15]:
    print(finding.to_text())
    if finding.remedy:
        print('         -> %s' % finding.remedy)


In [ ]:
!opslab daxlint --list-rules


## Your own extract

Every subcommand takes a CSV. The event log needs a case id, an activity, a
timestamp and optionally a resource; the case table needs a case id, an
arrival time, a duration in working hours, a resolved flag and an SLA target.
Upload yours to Colab and point `opslab try` at it:

```bash
opslab try --events my_events.csv --cases my_cases.csv --out report
```

The schema is documented in
[docs/using-your-own-data.md](https://github.com/pr1317/opslab2/blob/main/docs/using-your-own-data.md),
and the reasoning behind each estimator in
[docs/methods.md](https://github.com/pr1317/opslab2/blob/main/docs/methods.md).